# Visualization Toolkit
# 可视化工具包

PipelineTS provides a comprehensive visualization module with **automatic Chinese font support**.
PipelineTS 提供全面的可视化模块，**自动支持中文字体**。

This tutorial covers:
本教程涵盖：

1. **Chinese font configuration / 中文字体配置**
2. **Single-series & multi-series plotting / 单序列与多序列绑图**
3. **Forecast visualization / 预测可视化**
4. **Leaderboard charts / 排行榜图表**
5. **Model comparison overlay / 多模型对比叠加**
6. **Residual diagnostics / 残差诊断**
7. **ACF / PACF analysis / 自相关分析**
8. **Time series decomposition / 时间序列分解**
9. **Train/test split visualization / 训练测试集分割可视化**
10. **TSPlotter class / TSPlotter 类**
11. **Pipeline integration / 管道集成**

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Prepare example data / 准备示例数据
np.random.seed(42)
n = 200
dates = pd.date_range(start='2020-01-01', periods=n, freq='D')
values = 50 + 10 * np.sin(np.linspace(0, 6 * np.pi, n)) + np.random.randn(n) * 2
data = pd.DataFrame({'date': dates, 'value': values})

print(f"Data shape / 数据形状: {data.shape}")
data.head()

## 1. Chinese Font Configuration / 中文字体配置

matplotlib does not natively support Chinese characters. PipelineTS auto-detects and configures Chinese fonts.

matplotlib 默认不支持中文字符。PipelineTS 自动检测并配置中文字体。

In [ ]:
from PipelineTS.plot import configure_chinese_font

# Auto-detect and configure Chinese font
# 自动检测并配置中文字体
font_name = configure_chinese_font()
print(f"Detected font / 检测到的字体: {font_name}")

## 2. Single-Series Plotting / 单序列绑图

`plot_series()` visualizes time series data with automatic date formatting and bilingual labels.

`plot_series()` 可视化时间序列数据，自动日期格式化和双语标签。

In [ ]:
from PipelineTS.plot import plot_series

# Chinese labels (default) / 中文标签（默认）
plot_series(data, time_col='date', target_col='value', title='销量趋势', lang='zh')

In [ ]:
# English labels / 英文标签
plot_series(data, time_col='date', target_col='value', title='Sales Trend', lang='en')

### Multi-Series Panel / 多序列面板

When `id_col` is provided, `plot_series()` creates a grid of subplots.

提供 `id_col` 时，`plot_series()` 创建子图网格。

In [ ]:
# Create multi-series (panel) data / 创建多序列（面板）数据
panel_dfs = []
for sid in ['Store_A', 'Store_B', 'Store_C', 'Store_D']:
    offset = np.random.uniform(30, 70)
    amp = np.random.uniform(5, 15)
    vals = offset + amp * np.sin(np.linspace(0, 6 * np.pi, n)) + np.random.randn(n) * 2
    panel_dfs.append(pd.DataFrame({'date': dates, 'value': vals, 'store_id': sid}))
panel_data = pd.concat(panel_dfs, ignore_index=True)

plot_series(panel_data, time_col='date', target_col='value',
            id_col='store_id', title='各门店销量趋势', lang='zh')

## 3. Forecast Visualization / 预测可视化

`plot_forecast()` shows historical data and forecast with prediction intervals.

`plot_forecast()` 展示历史数据和带预测区间的预测值。

In [ ]:
from PipelineTS.plot import plot_forecast
from PipelineTS.ml_model import LightGBMModel

# Train a model / 训练模型
model = LightGBMModel(
    time_col='date', target_col='value', lags=12,
    quantile=0.9, n_estimators=200, verbose=-1
)
model.fit(data)
pred = model.predict(20)

# Plot forecast with prediction intervals
# 绘制带预测区间的预测图
plot_forecast(
    data, pred,
    time_col='date', target_col='value',
    history_tail=60,  # Show last 60 history points / 仅展示最后 60 个历史点
    title='LightGBM 预测结果',
    lang='zh',
)

## 4. Leaderboard Charts / 排行榜图表

After running `ModelPipeline`, visualize the model leaderboard.

运行 `ModelPipeline` 后，可视化模型排行榜。

In [ ]:
from PipelineTS.pipeline import ModelPipeline
from PipelineTS.plot import plot_leaderboard, plot_leaderboard_detail

# Train pipeline / 训练管道
pipeline = ModelPipeline(
    time_col='date', target_col='value', lags=12,
    include_models=['lightgbm', 'xgboost', 'catboost'],
    quantile=0.9, cv=2
)
leaderboard = pipeline.fit(data)
leaderboard

In [ ]:
# Simple leaderboard bar chart / 简单排行榜柱状图
plot_leaderboard(leaderboard, title='模型排行榜', lang='zh')

In [ ]:
# Detailed leaderboard with training/eval costs / 详细排行榜含训练/评估耗时
plot_leaderboard_detail(leaderboard, title='模型详细排行', lang='zh')

## 5. Model Comparison / 多模型预测对比

Overlay predictions from multiple models on the same chart.

在同一图表上叠加多个模型的预测结果。

In [ ]:
from PipelineTS.plot import plot_model_comparison

# Get predictions from different models / 获取不同模型的预测
models = leaderboard['model'].values
predictions = {}
for m in models:
    predictions[m] = pipeline.predict(20, model_name=m)

plot_model_comparison(
    data, predictions,
    time_col='date', target_col='value',
    history_tail=30,
    title='模型预测对比',
    lang='zh',
)

## 6. Residual Diagnostics / 残差诊断

4-panel diagnostic plot: residual time series, histogram, Q-Q, and ACF.

四面板诊断图：残差时序、直方图、Q-Q 图和 ACF 图。

In [ ]:
from PipelineTS.plot import plot_residuals

# Use the best model to generate in-sample predictions for diagnostics
# 使用最佳模型生成样本内预测用于诊断
best_pred = pipeline.predict(20)

# For demonstration, generate synthetic residuals
# 演示用：生成合成残差
y_true = data['value'].values[-50:]
y_pred = y_true + np.random.randn(50) * 2

plot_residuals(
    y_true, y_pred,
    time_index=data['date'].values[-50:],
    title='残差分析',
    lang='zh',
)

## 7. ACF / PACF Analysis / 自相关分析

Identify autocorrelation patterns to guide model selection.

识别自相关模式以指导模型选择。

In [ ]:
from PipelineTS.plot import plot_acf_pacf

plot_acf_pacf(
    data['value'].values,
    max_lags=30,
    title='自相关与偏自相关分析',
    lang='zh',
)

## 8. Time Series Decomposition / 时间序列分解

Decompose time series into trend, seasonal, and residual components.

将时间序列分解为趋势、季节性和残差分量。

In [ ]:
from PipelineTS.plot import plot_decomposition

plot_decomposition(
    data,
    time_col='date', target_col='value',
    period=None,       # Auto-detect / 自动检测
    model='additive',  # 'additive' or 'multiplicative'
    title='时间序列分解',
    lang='zh',
)

## 9. Train/Test Split Visualization / 训练测试集分割可视化

Visualize the partition between training and testing data.

可视化训练数据和测试数据的分割。

In [ ]:
from PipelineTS.plot import plot_train_test_split

train_data = data.iloc[:-30]
test_data = data.iloc[-30:]

plot_train_test_split(
    train_data, test_data,
    time_col='date', target_col='value',
    title='训练集 / 测试集分割',
    lang='zh',
)

## 10. TSPlotter Class / TSPlotter 类

Use `TSPlotter` for a reusable plotting interface that binds `time_col`, `target_col`, and `lang` once.

使用 `TSPlotter` 作为可复用的绑图接口，一次绑定 `time_col`、`target_col` 和 `lang`。

In [ ]:
from PipelineTS.plot import TSPlotter

plotter = TSPlotter(time_col='date', target_col='value', lang='zh')

# No need to pass time_col/target_col/lang each time
# 无需每次都传递 time_col/target_col/lang
plotter.plot_series(data, title='使用 TSPlotter')

In [ ]:
# TSPlotter can also do multi-series, forecast, decomposition, etc.
# TSPlotter 也可以进行多序列、预测、分解等绑图
plotter.plot_forecast(data, pred, history_tail=60, title='TSPlotter 预测图')

In [ ]:
plotter.plot_decomposition(data, title='TSPlotter 时序分解')

## 11. Pipeline Integration / 管道集成

`ModelPipeline` and `SmartRouter` have built-in `plot()` and `plot_leaderboard()` methods.

`ModelPipeline` 和 `SmartRouter` 内置 `plot()` 和 `plot_leaderboard()` 方法。

In [ ]:
# One-line forecast plot from the best model / 一行代码绘制最佳模型预测图
pipeline.plot(n=20, history_tail=60, lang='zh')

In [ ]:
# One-line leaderboard chart / 一行代码绘制排行榜
pipeline.plot_leaderboard(lang='zh')

In [ ]:
# Use a specific model / 指定模型
pipeline.plot(n=20, model_name='xgboost', history_tail=60, lang='en')

## 12. Color Palette / 颜色调色板

All plot functions use a modern, accessible color palette. You can import and customize it.

所有绑图函数使用现代、易读的颜色调色板。你可以导入并自定义。

In [ ]:
from PipelineTS.plot import COLORS, MODEL_COLORS

print("Named colors / 命名颜色:")
for k, v in COLORS.items():
    print(f"  {k}: {v}")

print(f"\nModel palette ({len(MODEL_COLORS)} colors) / 模型调色板:")
print(MODEL_COLORS)

## 13. Saving Figures / 保存图表

All plot functions return a `matplotlib.figure.Figure` object. Set `show=False` to prevent display and save to file.

所有绑图函数返回 `matplotlib.figure.Figure` 对象。设置 `show=False` 可防止显示并保存到文件。

In [ ]:
# Save figure to file / 保存图表到文件
fig = plot_forecast(
    data, pred,
    time_col='date', target_col='value',
    history_tail=60,
    title='保存到文件的预测图',
    show=False,  # Don't display / 不显示
)

# fig.savefig('forecast.png', dpi=150, bbox_inches='tight')
# fig.savefig('forecast.pdf', bbox_inches='tight')
print(f"Figure size / 图表尺寸: {fig.get_size_inches()}")
print("Uncomment the lines above to actually save. / 取消注释上面的行即可保存。")

## Summary / 总结

| Function / 函数 | Description / 描述 |
|---|---|
| `configure_chinese_font()` | Auto-detect Chinese font / 自动检测中文字体 |
| `plot_series()` | Single or multi-series / 单序列或多序列 |
| `plot_forecast()` | Actual vs forecast + intervals / 实际值 vs 预测值 + 区间 |
| `plot_leaderboard()` | Model ranking bar chart / 模型排名柱状图 |
| `plot_leaderboard_detail()` | Leaderboard + cost breakdown / 排行榜 + 耗时 |
| `plot_model_comparison()` | Multi-model overlay / 多模型叠加 |
| `plot_residuals()` | 4-panel diagnostics / 四面板诊断 |
| `plot_acf_pacf()` | ACF + PACF / 自相关 + 偏自相关 |
| `plot_decomposition()` | Trend + seasonal + residual / 趋势 + 季节性 + 残差 |
| `plot_train_test_split()` | Data partition / 数据分割 |
| `TSPlotter()` | Reusable class / 可复用类 |
| `pipeline.plot()` | One-line pipeline plot / 一行代码管道绘图 |
| `pipeline.plot_leaderboard()` | One-line leaderboard / 一行代码排行榜 |